# Cosmos 3 Dataset Explorer (Google Colab)

Notebook này dùng **metadata-first + streaming** để xem schema, sample, phân bố và preview mà không tải hàng chục TB. Chạy lần lượt từ trên xuống. Với dataset gated, hãy chấp nhận điều khoản trên Hugging Face và thêm secret `HF_TOKEN` trong Colab.

## Research objective and scope

Notebook này trả lời năm câu hỏi trước khi team lựa chọn dữ liệu:

1. Dataset nào được Cosmos 3 technical report xác nhận là dữ liệu train, dataset nào chỉ là supporting release hoặc benchmark?
2. Mỗi bộ cung cấp modality và supervision gì: RGB, video, depth, segmentation, boxes, physics, text hay action?
3. Dữ liệu gần Industrial AI đến mức nào và còn thiếu phần nào của nhà máy thực?
4. Bộ nào public, gated, internal hoặc chưa xác định license/phạm vi phát hành?
5. Bộ nào nên dùng cho foundation learning, industrial adaptation, benchmark và held-out validation?

**Boundary:** notebook chỉ phân tích dữ liệu public có thể kiểm chứng. Nó không đại diện cho toàn bộ 767M ảnh, 347.7M video hay các nguồn nội bộ của Cosmos 3. Preview trên dataset card cũng không phải là một mẫu ngẫu nhiên đủ đại diện cho toàn corpus.

## Evidence labels used in this review

| Label | Interpretation |
|---|---|
| **Used** | Cosmos 3 report explicitly says the source/mixture was used for training. |
| **Used + released** | The source is named in training and a public NVIDIA/original release exists. |
| **Supporting release** | It appears in the Cosmos 3 ecosystem, but that alone does not prove every released sample was consumed. |
| **Benchmark** | Used to evaluate or judge quality; it must not be silently mixed into training. |
| **Internal/unavailable** | The exact source, transform or subset cannot currently be reproduced publicly. |

Dataset cards may be updated after the paper. If paper counts and current release counts differ, preserve both values and the source revision instead of merging them.

In [ ]:
%pip install -q -U huggingface_hub datasets pandas pyarrow matplotlib seaborn ipywidgets


In [ ]:
import base64, html, json, os, random, re
from itertools import islice
from pathlib import Path
from urllib.parse import quote
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image, Video, HTML
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, login

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face authentication:', 'enabled' if HF_TOKEN else 'public-only')
WORK = Path('/content/cosmos3_dataset_explorer')
WORK.mkdir(parents=True, exist_ok=True)


## 1. Dataset catalog
`full_size` chỉ để cảnh báo. Notebook không tự tải raw corpus.

In [ ]:
CATALOG = [
 {'repo':'nvidia/Cosmos-HumanEval-v1','role':'benchmark','full_size':'1.15 MB','access':'public'},
 {'repo':'nvidia/PhysicalAI-Traffic-Anomaly-Reasoning','role':'reasoner/industry','full_size':'59 MB annotations + ~150 GB source video','access':'public/mixed licenses'},
 {'repo':'nvidia/PhysicalAI-Spatial-Intelligence-Warehouse','role':'reasoner/warehouse','full_size':'multi-GB','access':'gated CC-BY-4.0'},
 {'repo':'nvidia/Cosmos3-DROID','role':'generator/action','full_size':'~758 GB','access':'public OpenMDW-1.1'},
 {'repo':'nvidia/PhysicalAI-WorldModel-Synthetic-Physical-Interaction-Scenes','role':'generator/physics','full_size':'~16.4 TB','access':'public OpenMDW-1.1'},
 {'repo':'nvidia/PhysicalAI-WorldModel-Synthetic-Embodied-Robot-Scenes','role':'generator/robotics','full_size':'~2.08 TB','access':'public OpenMDW-1.1'},
 {'repo':'nvidia/PhysicalAI-WorldModel-Synthetic-Autonomous-Driving-Scenarios','role':'generator/AV','full_size':'~8.24 TB','access':'public OpenMDW-1.1'},
 {'repo':'nvidia/PhysicalAI-WorldModel-Synthetic-Digital-Human-Scenes','role':'generator/human','full_size':'very large','access':'public OpenMDW-1.1'},
 {'repo':'nvidia/PhysicalAI-WorldModel-Synthetic-Warehouse-Operations-Scenes','role':'generator/industry','full_size':'~18.57 TiB','access':'public OpenMDW-1.1'},
 {'repo':'IPEC-COMMUNITY/EO-Data1.5M','role':'reasoner/robotics','full_size':'~201 GB','access':'public Apache-2.0'},
 {'repo':'nexar-ai/nexar_collision_prediction','role':'reasoner/AV','full_size':'~31.4 GB','access':'public/source terms'},
]
catalog_df = pd.DataFrame(CATALOG)
display(catalog_df)


## Dataset capability matrix
Ma trận này cho thấy các bộ public bổ sung cho nhau; không có một bộ nào tự nó bao phủ toàn bộ Industrial World Model. Giá trị `1` nghĩa là modality/capability được cung cấp trực tiếp hoặc là supervision chính.

In [ ]:
PROFILE = pd.DataFrame([
 {'dataset':'HUE','real':0,'video':1,'depth':0,'seg/box':0,'text/reason':1,'physics':1,'action':0,'industrial':0},
 {'dataset':'TAR','real':1,'video':1,'depth':0,'seg/box':1,'text/reason':1,'physics':0,'action':0,'industrial':1},
 {'dataset':'Warehouse Spatial','real':0,'video':0,'depth':1,'seg/box':1,'text/reason':1,'physics':0,'action':0,'industrial':1},
 {'dataset':'DROID','real':1,'video':1,'depth':1,'seg/box':0,'text/reason':1,'physics':0,'action':1,'industrial':1},
 {'dataset':'PhyxSim','real':0,'video':1,'depth':1,'seg/box':1,'text/reason':1,'physics':1,'action':0,'industrial':1},
 {'dataset':'RobotSim','real':0,'video':1,'depth':0,'seg/box':0,'text/reason':1,'physics':1,'action':1,'industrial':1},
 {'dataset':'DriveSim','real':0,'video':1,'depth':0,'seg/box':0,'text/reason':1,'physics':0,'action':0,'industrial':0},
 {'dataset':'SynHuman','real':0,'video':1,'depth':1,'seg/box':0,'text/reason':0,'physics':0,'action':0,'industrial':1},
 {'dataset':'SDG-Warehouse','real':0,'video':1,'depth':1,'seg/box':1,'text/reason':0,'physics':1,'action':0,'industrial':1},
 {'dataset':'EO-Data','real':1,'video':1,'depth':0,'seg/box':1,'text/reason':1,'physics':0,'action':1,'industrial':1},
 {'dataset':'Nexar','real':1,'video':1,'depth':0,'seg/box':0,'text/reason':0,'physics':0,'action':0,'industrial':0},
]).set_index('dataset')
plt.figure(figsize=(12,6))
sns.heatmap(PROFILE,annot=True,cmap=['#eeeeee','#1565c0'],cbar=False,linewidths=.5,linecolor='white')
plt.title('Public data capability coverage'); plt.xlabel('Provided capability'); plt.ylabel('')
plt.tight_layout(); plt.show()
display(PROFILE)


### How to interpret the matrix

- **Synthetic datasets** dominate dense geometry and physics because simulators expose ground truth cheaply.
- **Real datasets** are essential for appearance, sensor noise, human behavior and deployment validity.
- **Action data** is concentrated in DROID, RobotSim and EO-Data; warehouse safety video alone does not teach intervention or control.
- **Benchmarks such as HUE** should remain isolated from training splits to preserve honest evaluation.
- `industrial=1` means industrial-adjacent relevance, not proof that the data matches a specific production line.

## 2. Inspect repository schema and files
Đổi `SELECTED_REPO` để xem dataset khác. Cell chỉ gọi API metadata.

In [ ]:
SELECTED_REPO = 'nvidia/PhysicalAI-WorldModel-Synthetic-Warehouse-Operations-Scenes'
api = HfApi(token=HF_TOKEN)
info = api.dataset_info(SELECTED_REPO, files_metadata=True)
rows = []
for f in info.siblings:
    size = getattr(f, 'size', None)
    rows.append({'path': f.rfilename, 'size_MiB': None if size is None else round(size/2**20, 3)})
files_df = pd.DataFrame(rows)
print('repo:', SELECTED_REPO, '| files:', len(files_df), '| revision:', info.sha)
display(files_df.head(30))
print('known size GiB:', round(files_df.size_MiB.fillna(0).sum()/1024, 3))


## 3. Cosmos-HumanEval: xem prompt và câu hỏi đánh giá
Bộ này nhỏ nên tải toàn bộ JSON.

In [ ]:
hue_path = hf_hub_download('nvidia/Cosmos-HumanEval-v1', 'hue-v1p2-t2v-public.json', repo_type='dataset', token=HF_TOKEN)
hue = json.load(open(hue_path, encoding='utf-8'))
samples = hue['samples']
sample = random.choice(samples)
questions = []
for key, value in sample.items():
    if re.fullmatch(r'question_\d+', key):
        m = re.match(r'\[([^]]+)\]\[([^]]+)\]\s*(.*)', value)
        questions.append({'dimension': m.group(1) if m else '', 'subtype': m.group(2) if m else '', 'question': m.group(3) if m else value})
display(Markdown('### Prompt\n' + sample['prompt']))
display(pd.DataFrame(questions))
all_dims=[]
for s in samples:
    for k,v in s.items():
        if re.fullmatch(r'question_\d+', k):
            m=re.match(r'\[([^]]+)\]',v)
            if m: all_dims.append(m.group(1))
pd.Series(all_dims).value_counts().plot.bar(title='HUE question dimensions'); plt.show()


## 4. TAR: xem annotation thật theo từng task
Cell tải annotation (~59 MB), không tải ~150 GB video nguồn.

In [ ]:
tar_dir = snapshot_download(
    'nvidia/PhysicalAI-Traffic-Anomaly-Reasoning', repo_type='dataset', token=HF_TOKEN,
    allow_patterns=['README.md','DOWNLOADING.md','train/*.json','test/*.json','test/*.csv']
)
task_rows=[]
for p in sorted(Path(tar_dir).glob('train/*.json')):
    obj=json.load(open(p,encoding='utf-8'))
    items=obj.get('items',[])
    task_rows.append({'task':p.stem,'samples':len(items),'mean_answer_chars':round(sum(len(str(x.get('answer',''))) for x in items)/max(1,len(items)),1)})
task_df=pd.DataFrame(task_rows)
display(task_df)
fig,axes=plt.subplots(1,2,figsize=(13,4))
sns.barplot(data=task_df,x='samples',y='task',ax=axes[0],palette='crest'); axes[0].set_title('TAR samples by task')
sns.barplot(data=task_df,x='mean_answer_chars',y='task',ax=axes[1],palette='flare'); axes[1].set_title('Mean answer length')
plt.tight_layout(); plt.show()
task_file = Path(tar_dir)/'train/open_qa.json'
item = random.choice(json.load(open(task_file,encoding='utf-8'))['items'])
display(Markdown(f"**Video ID:** `{item.get('video_id')}`\n\n**Question:** {item.get('question')}\n\n**Answer:** {item.get('answer')}\n\n**Reasoning:** {item.get('reasoning','')}"))


## 5. SDG-Warehouse: xem index của 122k clips
Chỉ tải hai Parquet index vài MB, không tải shard video 5 GB.

In [ ]:
WH='nvidia/PhysicalAI-WorldModel-Synthetic-Warehouse-Operations-Scenes'
runs_path=hf_hub_download(WH,'metadata/runs.parquet',repo_type='dataset',token=HF_TOKEN)
clips_path=hf_hub_download(WH,'metadata/clips.parquet',repo_type='dataset',token=HF_TOKEN)
runs=pd.read_parquet(runs_path); clips=pd.read_parquet(clips_path)
print('runs:',len(runs),'clips:',len(clips))
display(runs.head(3)); display(clips.head(3))
if 'scenario' in clips:
    clips['scenario'].value_counts().plot.bar(title='SDG-Warehouse clips by scenario'); plt.ylabel('clips'); plt.show()
if 'camera_alias' in clips:
    display(clips['camera_alias'].value_counts().head(20).rename('clips').to_frame())
display(clips.sample(min(10,len(clips)), random_state=7))


## 6. RobotSim: xem preview GIF thực tế
Preview chỉ vài MB. Có thể đổi tên file sang collision, DreamZero, Simulario hoặc motion theo file inventory.

In [ ]:
ROBOT='nvidia/PhysicalAI-WorldModel-Synthetic-Embodied-Robot-Scenes'
gif_name='PhysicalAI-Cosmos-SDG-RobotSim-manipulation-mimicgen.gif'
gif_path=hf_hub_download(ROBOT,gif_name,repo_type='dataset',token=HF_TOKEN)
display(Image(filename=gif_path))


## 7. Visual dashboard: quy mô dataset
Trục X dùng log scale vì dataset nhỏ nhất và lớn nhất chênh nhau hàng triệu lần.

In [ ]:
SIZE_GB = {
 'Cosmos-HumanEval':0.00115, 'TAR annotations':0.059, 'Nexar':31.4,
 'EO-Data-1.5M':201, 'Cosmos3-DROID':758, 'RobotSim':2080,
 'DriveSim':8240, 'PhyxSim':16400, 'SDG-Warehouse':19015,
}
size_df=pd.DataFrame({'dataset':SIZE_GB.keys(),'GB':SIZE_GB.values()}).sort_values('GB')
plt.figure(figsize=(11,5))
ax=sns.barplot(data=size_df,x='GB',y='dataset',palette='viridis')
ax.set_xscale('log'); ax.set_title('Approximate public dataset size (log scale)')
ax.set_xlabel('GB, logarithmic'); ax.set_ylabel('')
for container in ax.containers: ax.bar_label(container,fmt='%.3g',padding=3)
plt.tight_layout(); plt.show()


## 8. Visual gallery: dữ liệu trông như thế nào
Các file dưới đây là preview chính thức, nhỏ hơn shard train rất nhiều.

In [ ]:
from PIL import Image as PILImage

def hub_asset(repo, path):
    return hf_hub_download(repo,path,repo_type='dataset',token=HF_TOKEN)

display(Markdown('### PhyxSim — rigid-body physics'))
display(Image(filename=hub_asset('nvidia/PhysicalAI-WorldModel-Synthetic-Physical-Interaction-Scenes','PhysicalAI-Cosmos-SDG-PhysxSim.gif')))

display(Markdown('### SynHuman — human motion and camera trajectories'))
display(Image(filename=hub_asset('nvidia/PhysicalAI-WorldModel-Synthetic-Digital-Human-Scenes','assets/hugging_face_gif_01_small.gif')))

display(Markdown('### SDG-Warehouse — four industrial scenarios'))
wh_assets=['assets/clip_nearmiss.webp','assets/clip_fire.webp','assets/clip_forklift_collision.webp','assets/clip_box_pickup.webp']
for title,path in zip(['Forklift near-miss','Fire evacuation','Forklift collision','Box pickup'],wh_assets):
    display(Markdown('**'+title+'**'))
    webp_path=hub_asset(WH,path)
    webp_b64=base64.b64encode(Path(webp_path).read_bytes()).decode('ascii')
    display(HTML(f'<img src="data:image/webp;base64,{webp_b64}" style="max-width:700px;width:100%;height:auto">'))


## 9. Multimodal supervision gallery
So sánh RGB với depth, segmentation, shaded segmentation và edge map của cùng loại scene.

In [ ]:
modalities=[
 ('RGB scenario','assets/scenario_nearmiss.jpg'),
 ('Metric depth','assets/nearmiss_depth.jpg'),
 ('Instance segmentation','assets/nearmiss_segmentation.jpg'),
 ('Shaded segmentation','assets/nearmiss_shaded_seg.jpg'),
 ('Canny edges','assets/nearmiss_edges.jpg'),
]
fig,axes=plt.subplots(1,len(modalities),figsize=(20,4))
for ax,(title,path) in zip(axes,modalities):
    im=PILImage.open(hub_asset(WH,path))
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.suptitle('SDG-Warehouse paired supervision'); plt.tight_layout(); plt.show()


## 10. Warehouse Spatial Intelligence: RGB + depth sample
Dataset này gated. Nếu cell báo 401/403, accept điều khoản trên Hugging Face rồi thêm `HF_TOKEN` vào Colab Secrets.

In [ ]:
SPATIAL='nvidia/PhysicalAI-Spatial-Intelligence-Warehouse'
try:
    rgb=hub_asset(SPATIAL,'train_sample/images/001511.png')
    depth=hub_asset(SPATIAL,'train_sample/depths/001511_depth.png')
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    axes[0].imshow(PILImage.open(rgb)); axes[0].set_title('Warehouse RGB')
    axes[1].imshow(PILImage.open(depth),cmap='magma'); axes[1].set_title('Warehouse depth')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
except Exception as exc:
    print(type(exc).__name__,exc)


## 11. EO-Data-1.5M: annotation and grounding examples

In [ ]:
EO='IPEC-COMMUNITY/EO-Data1.5M'
eo_assets=[
 ('Dataset example','.assets/data_example.png'),
 ('Point grounding','.assets/point_vis_eg.png'),
 ('Referring grounding','.assets/referring_vis_eg.png'),
 ('Trajectory','.assets/traj_vis_eg.png'),
]
fig,axes=plt.subplots(2,2,figsize=(14,10))
for ax,(title,path) in zip(axes.flat,eo_assets):
    ax.imshow(PILImage.open(hub_asset(EO,path))); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


## 12. Generic streaming preview
Dùng cho repo có Dataset Viewer/Parquet. Media decoding được tắt để tránh tải video lớn. Nếu repo gated, cần `HF_TOKEN` và phải accept access trên trang dataset trước.

In [ ]:
from datasets import load_dataset

def hf_video_player(media, width=760):
    uri = media.get('path','') if isinstance(media,dict) else str(media)
    match = re.match(r'hf://datasets/([^@]+)@([^/]+)/(.+)', uri)
    if not match:
        return Markdown(f'Cannot convert video URI: `{uri}`')
    repo, revision, relative = match.groups()
    url = f'https://huggingface.co/datasets/{repo}/resolve/{revision}/{quote(relative,safe="/")}'
    return HTML(
        f'<video controls preload="metadata" width="{width}" '
        f'style="max-width:100%;background:#111" src="{html.escape(url)}"></video>'
    )

def stream_preview(repo, split='train', n=3, play_video=True):
    try:
        ds=load_dataset(repo, split=split, streaming=True, token=HF_TOKEN)
        try: ds=ds.decode(False)
        except Exception: pass
        rows=list(islice(ds,n))
        for i,row in enumerate(rows):
            print(f'--- sample {i} ---')
            display({k:(str(v)[:500]+'...' if len(str(v))>500 else v) for k,v in row.items()})
            if play_video:
                for value in row.values():
                    path=value.get('path','') if isinstance(value,dict) else ''
                    if path.lower().endswith(('.mp4','.webm','.mov')):
                        display(hf_video_player(value))
                        break
        return rows
    except Exception as exc:
        print(type(exc).__name__, exc)
        print('Dataset có thể gated, không hỗ trợ streaming, hoặc cần chọn config/split khác.')
        return []

# Ví dụ nhỏ. Bỏ comment từng dòng để thử:
rows = stream_preview('nvidia/Cosmos3-DROID', n=2, play_video=True)
# rows = stream_preview('IPEC-COMMUNITY/EO-Data1.5M', n=2)
# rows = stream_preview('nexar-ai/nexar_collision_prediction', n=2)
# rows = stream_preview('nvidia/PhysicalAI-Spatial-Intelligence-Warehouse', n=2)


## Cosmos 3 data vs. Industrial AI data

| Dimension | Cosmos/Physical-AI public data | Typical public Industrial data | Gap to close with company data |
|---|---|---|---|
| Objective | General world understanding, simulation and action | Narrow inspection/anomaly task | Define the exact operational task and success condition |
| Temporal coverage | Large video, multi-view and 24–30 FPS | Often static product images | Continuous machine cycles and pre-event/post-event windows |
| Dense labels | Depth, masks, 2D/3D boxes, cameras, simulator physics | Defect class and pixel mask | Machine state, causal event, intervention and outcome |
| Realism | Large but much of the specialist data is synthetic | Small but often captured from real products | Factory-specific optics, materials, vibration, dust and lighting |
| Sensors | Mainly RGB/video/depth/action | Mainly RGB; some 3D/thermal | PLC, torque, current, vibration, acoustic and synchronized timestamps |
| Domain specificity | Generic warehouse, robot, human and vehicle assets | Specific parts/products but few environments | Company machines, layouts, SOPs, PPE and failure taxonomy |
| Action consequence | Present in robot-policy streams | Usually absent | Observation → action → machine response → outcome |

**Interpretation:** Cosmos data is suitable for foundation learning. It cannot replace held-out real industrial data or company-specific adaptation. Synthetic expansion should only target a measured coverage gap and must pass an A/B admission test on untouched real data.

## Proposed data role separation

1. **Foundation:** general Cosmos/Physical-AI visual, temporal, physical and action data.
2. **Industrial adaptation:** public real industrial inspection, embodied reasoning and anomaly data.
3. **Company specialization:** real machines, sites, sensors, SOPs, events and operator/robot actions.
4. **Validation:** site/scene/time-separated real data that is never used for training or prompt tuning.
5. **Synthetic gap filling:** generated only after a missing condition has been measured; admitted only when real held-out metrics improve without safety regression.

## Questions for team discussion

- What is our first Industrial-AI boundary: inspection, warehouse safety, robot manipulation, predictive maintenance, or multi-task?
- Which output must the model produce: answer, region/mask, trajectory, event forecast, action, or future video?
- Which real company modalities are available and legally usable? Are timestamps synchronized?
- What rare but safety-critical failures must exist in the test set?
- Which source/site/time groups must be isolated to prevent leakage?
- What minimum real-data metric must synthetic data improve before admission?
- Which licenses permit the intended commercial research and deployment?

A team decision on these questions should happen before bulk download or new synthetic-data generation.

## Primary references

- [Cosmos 3 technical report](https://research.nvidia.com/labs/cosmos-lab/cosmos3/technical-report.pdf)
- [Official Cosmos 3 Hugging Face collection](https://huggingface.co/collections/nvidia/cosmos3)
- [PhysicalAI Spatial Intelligence Warehouse](https://huggingface.co/datasets/nvidia/PhysicalAI-Spatial-Intelligence-Warehouse)
- [SDG-Warehouse](https://huggingface.co/datasets/nvidia/PhysicalAI-WorldModel-Synthetic-Warehouse-Operations-Scenes)
- [Traffic Anomaly Reasoning](https://huggingface.co/datasets/nvidia/PhysicalAI-Traffic-Anomaly-Reasoning)
- [DROID official project](https://droid-dataset.github.io/)
- [EO-Data-1.5M](https://huggingface.co/datasets/IPEC-COMMUNITY/EO-Data1.5M)
- [MVTec LOCO AD](https://www.mvtec.com/research-teaching/datasets/mvtec-loco-ad) and [Real-IAD](https://realiad4ad.github.io/Real-IAD/) for comparison with real industrial anomaly data.

## 13. Export exploration report
Tải JSON này về để đưa vào báo cáo dataset.

In [ ]:
report={
 'catalog':CATALOG,
 'selected_repo':SELECTED_REPO,
 'selected_repo_revision':info.sha,
 'selected_repo_files':len(files_df),
 'hue_samples':len(samples),
 'tar_tasks':task_rows,
 'sdg_warehouse_runs':len(runs),
 'sdg_warehouse_clips':len(clips),
 'capability_matrix':PROFILE.reset_index().to_dict(orient='records'),
 'conclusions':[
   'Cosmos public data is suitable for foundation learning but does not replace real industrial validation.',
   'No single public dataset covers visual, temporal, physical, action and industrial specificity together.',
   'Synthetic data should be admitted only after an A/B test on untouched real data.'
 ],
 'policy':'metadata-first; no bulk raw download'
}
out=WORK/'cosmos3_dataset_exploration_report.json'
out.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(out)
try:
    from google.colab import files
    files.download(str(out))
except Exception:
    pass
